# Day 21 - Week 3 Review: Integrated Document Analyzer

**Sehar Andleeb - AI Engineer Intern @ Xeven Solutions**  
Repo: `ai-internship-xeven-2026`

This is a *consolidation* day. One integrated app combines the Week 3 building blocks into a single pipeline:

| Week 3 day | Concept | Where it shows up here |
|---|---|---|
| Day 17 | Embeddings + semantic search (MiniLM, cosine) | `embeddings_index.py` |
| Day 18 | Smart chunking (RecursiveCharacterTextSplitter) | `chunker.py` |
| Day 20 | Pydantic v2 structured extraction | `entity_extraction.py` |

**Pipeline:** load -> chunk -> embed/index -> semantic search -> structured extraction -> accuracy vs gold -> JSON report.

**Learning objectives achieved:** wire independent components into one importable, PEP8, lint-clean package; tune parameters to the data instead of copying textbook numbers; and verify the wiring end-to-end with honest metrics.

> **Run modes.** This notebook runs OFFLINE (deterministic hashed embedder + offline extraction stub) so it executes with no model download and no API key. The real path (`all-MiniLM-L6-v2` via `HuggingFaceEmbeddings` + Groq `llama-3.3-70b-versatile` via `ChatGroq.with_structured_output`) lives behind `--live` / `use_offline=False`. FAISS itself is REAL in both modes.

## Architecture

```mermaid
flowchart LR
    A[PDF / text docs<br/>document_loader.py] --> B[RecursiveCharacterTextSplitter<br/>chunker.py<br/>size=400 overlap=60]
    B --> C[Embeddings<br/>embeddings_index.py<br/>MiniLM 384-d / offline hash]
    C --> D[(FAISS IndexFlatIP<br/>cosine)]
    Q[User query] --> C
    D --> E[Top-k chunks + scores]
    A --> F[Pydantic extraction<br/>entity_extraction.py<br/>ChatGroq.with_structured_output / offline stub]
    F --> G[Score vs gold<br/>micro P/R/F1]
    E --> H[analyzer.py<br/>JSON report]
    G --> H
```

`analyzer.py` orchestrates the flow; `run_demo.py` is the CLI entry point. Each module owns exactly one concern.

In [1]:
# Dependencies (installed once in .venv312 with uv):
#   uv venv .venv312 --python 3.12
#   uv pip install langchain langchain-community langchain-groq \
#     langchain-huggingface sentence-transformers faiss-cpu pydantic \
#     python-dotenv pypdf numpy scikit-learn matplotlib
# (faiss-cpu has no Python 3.13 wheel yet -> 3.12 env.)
# Notebooks never shell out to pip.
print('Dependencies ready.')

Dependencies ready.


In [2]:
import os
import sys

# Run from inside scripts/ so outputs land in scripts/outputs/
if os.path.basename(os.getcwd()) != 'scripts':
    os.chdir('scripts')
sys.path.insert(0, os.getcwd())

import analyzer
import chunker
import document_loader as dl
import embeddings_index as ei
import entity_extraction as ee
print('cwd:', os.getcwd())

cwd: /home/claude/day21/scripts


### 1. Load documents (auto-created sample corpus)

In [3]:
dl.create_sample_corpus('data')
docs = dl.load_corpus('data')
for d in docs:
    print(f"{d['filename']:>22}  {len(d['text']):>4} chars")
print('\nPDF text round-trips via pypdf:',
      'Globex Corporation' in
      next(x['text'] for x in docs if x['filename'].endswith('.pdf')))

        memo_cloud.txt   502 chars
     research_note.txt   419 chars
  contract_summary.pdf   152 chars

PDF text round-trips via pypdf: True


### 2. Chunking + parameter tuning

I did not copy a textbook chunk size. I swept `[256, 400, 512]` on the test query and chose the balance point (see `chunker.py` docstring).

In [4]:
rows = chunker.tune_chunk_size(
    docs, dl.TEST_QUERY, dl.EXPECTED_TOP_PHRASE)
for r in rows:
    print(r)
print(f"\nChosen: size={chunker.DEFAULT_CHUNK_SIZE}, "
      f"overlap={chunker.DEFAULT_CHUNK_OVERLAP}")
chunks = chunker.chunk_corpus(docs)
print('Total chunks:', len(chunks))

{'chunk_size': 256, 'chunk_overlap': 36, 'num_chunks': 7, 'top1_hit': True, 'top1_score': 0.6455}
{'chunk_size': 400, 'chunk_overlap': 57, 'num_chunks': 5, 'top1_hit': True, 'top1_score': 0.5202}
{'chunk_size': 512, 'chunk_overlap': 73, 'num_chunks': 3, 'top1_hit': True, 'top1_score': 0.4449}

Chosen: size=400, overlap=60
Total chunks: 5


### 3. Embeddings + FAISS semantic search

Vectors are L2-normalized and indexed with `faiss.IndexFlatIP`, so inner product = cosine similarity. The known-answer query should return the cloud-migration memo as top-1.

In [5]:
embedder = ei.get_embedder(use_offline=True)
index = ei.SemanticIndex(embedder).build(chunks)
hits = index.search(dl.TEST_QUERY, k=3)
print('Query:', dl.TEST_QUERY, '\n')
for i, h in enumerate(hits, 1):
    print(f"#{i} [{h['score']:.4f}] {h['source']}")
    print('   ', h['text'][:90].replace(chr(10), ' '), '...')
print('\nExpected phrase in top-1:',
      dl.EXPECTED_TOP_PHRASE in hits[0]['text'])

Query: What is the approved budget for the cloud migration project? 

#1 [0.5202] memo_cloud.txt
    Internal Memo - Cloud Migration Initiative  From: Ayesha Khan (Director of Engineering) To ...
#2 [0.3326] research_note.txt
    We benchmarked the all-MiniLM-L6-v2 embedding model against a larger encoder. The MiniLM m ...
#3 [0.1111] memo_cloud.txt
    Vendor evaluation will be led by Bilal Ahmed in coordination with Acme Cloud Services. Ple ...

Expected phrase in top-1: True


### 4. Structured extraction with Pydantic v2

Typed fields + a `@field_validator` (regex email check, no `EmailStr` dependency). Valid data constructs; invalid raises `ValidationError`.

In [6]:
from pydantic import ValidationError

doc0 = docs[0]
pred = ee.extract_entities(doc0['text'], doc0['gold'],
                           use_offline=True)
print('Extracted from', doc0['filename'], ':')
for k, v in pred.model_dump().items():
    print(f'  {k}: {v}')

try:
    ee.DocumentEntities(emails=['not-an-email'])
except ValidationError as exc:
    print('\nInvalid email correctly rejected:',
          exc.errors()[0]['msg'])

Extracted from memo_cloud.txt :
  people: ['Ayesha Khan']
  organizations: ['Acme Cloud Services', 'Unverified Holdings']
  emails: ['ayesha.khan@xeven.example.com', 'finance@xeven.example.com']
  dates: ['2026-03-14', '2026-04-01']
  monetary_amounts: ['$250,000']

Invalid email correctly rejected: Value error, Invalid email address: 'not-an-email'


### 5. Full pipeline -> JSON report + accuracy vs gold

Accuracy is computed against known gold labels. The offline stub injects one known miss + one false positive per document, so the metric is deliberately *not* a trivial 100%.

In [7]:
report = analyzer.analyze('data', 'outputs', use_offline=True)
path = analyzer.export_report(report, 'outputs')
ext = report['extraction']
print('Report:', path)
print('Chunks:', report['corpus']['num_chunks'])
print(f"micro-F1: {ext['micro_f1']}  "
      f"(P={ext['micro_precision']}, R={ext['micro_recall']})")
print('Approx input tokens:',
      report['cost_estimate']['approx_input_tokens'],
      '| charge $', report['cost_estimate']['charge_usd'])

Report: outputs/analysis_report.json
Chunks: 5
micro-F1: 0.8947  (P=0.85, R=0.9444)
Approx input tokens: 269 | charge $ 0.0


## Research comparison - RAG / document-analysis architecture

*Consulted 2026-06-15.* Lighter than usual since this is a review day; topic chosen because Week 4 is RAG & vector databases.

I wrote the **Claude** column and the **Article** column myself. The **ChatGPT** and **Gemini** columns are pre-filled with the cross-tool consensus and marked *(confirm)* - Sehar to verify against the live tools.

| Question | ChatGPT *(confirm)* | Gemini *(confirm)* | Claude | Article: Firecrawl, *Best Chunking Strategies for RAG 2026* |
|---|---|---|---|---|
| Default chunker for a first RAG build? | RecursiveCharacterTextSplitter as the reliable baseline | Recursive splitter first, then structure-aware splitters | Recursive splitter - it respects natural separators and is the right starting baseline before anything fancier | Recursive 512-token splitting ranked #1 (69% acc) in Vecta's Feb 2026 benchmark of 7 strategies |
| Chunk size / overlap? | ~512 tokens, 10-20% overlap, tune per corpus | 256-512 tokens with ~15% overlap | Tune to the data: I chose 400 chars / 60 overlap here because these are short single-fact memos; smaller chunks scored higher but fragment more | 512-token recursive + 10-20% overlap is the benchmark-validated default |
| Embedding model choice? | Start with MiniLM, upgrade if recall is weak | MTEB leaderboard as a guide, benchmark on your data | all-MiniLM-L6-v2 (384-d) is a strong, cheap, local default; validate on your own queries | MTEB scores don't predict domain performance - always benchmark on your own data |
| How to scale to production? | Managed vector DB + metadata + caching | Pinecone/managed store, hybrid search, eval harness | Move FAISS-local -> managed store (Pinecone, Week 4), add metadata filtering and an eval set; keep chunking/retrieval measurable | Metadata enrichment lifts QA accuracy ~15-25 points with no retrieval-architecture change; hierarchical (parent-child) chunking is the common production pattern |

Source: Firecrawl, *Best Chunking Strategies for RAG (and LLMs) in 2026* - https://www.firecrawl.dev/blog/best-chunking-strategies-rag

## Running the REAL pipeline (on my machine, `.venv312`)

```bash
cd day21/scripts
uv run python run_demo.py            # offline (this notebook's mode)
uv run python run_demo.py --live     # real MiniLM + real Groq
uv run python verify_pipeline.py     # 6 wiring assertions
```
`--live` needs `GROQ_API_KEY` in a local `.env` (read via `os.getenv` after `load_dotenv()`; never hard-coded) and downloads the MiniLM model (~80 MB) once on first run.